<a href="https://colab.research.google.com/github/Leanhchudang2511/baitaptrituenhantao/blob/main/d%E1%BB%B1_%C4%91o%C3%A1n_%C4%91%E1%BB%99_stress.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit pyngrok scikit-learn pandas numpy matplotlib seaborn openpyxl --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 34.2 MB/s eta 0:00:00


In [2]:
%%writefile streamlit_app.py
# -*- coding: utf-8 -*-

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import permutation_importance


# =========================
# CẤU HÌNH FONT
# =========================

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False


# =========================
# CẤU HÌNH TRANG
# =========================

st.set_page_config(
    page_title="Dự Đoán Mức Độ Căng Thẳng",
    page_icon="🧠",
    layout="wide"
)

st.markdown("""
<style>
.main-header {
    background: linear-gradient(135deg, #111827, #4338ca, #7c3aed);
    padding: 2rem;
    border-radius: 22px;
    text-align: center;
    margin-bottom: 1.5rem;
    box-shadow: 0px 8px 25px rgba(0,0,0,0.18);
}
.main-header h1 {
    color: white;
    font-size: 2.35rem;
    margin: 0;
}
.main-header p {
    color: #ede9fe;
    margin-top: .6rem;
    font-size: 1.05rem;
}
.result-low {
    background: linear-gradient(135deg, #dcfce7, #bbf7d0);
    border: 2px solid #16a34a;
    border-radius: 20px;
    padding: 2rem;
    text-align: center;
}
.result-high {
    background: linear-gradient(135deg, #fee2e2, #fecaca);
    border: 2px solid #dc2626;
    border-radius: 20px;
    padding: 2rem;
    text-align: center;
}
.feature-card {
    background: #f8fafc;
    border-left: 6px solid #7c3aed;
    padding: 1rem;
    border-radius: 14px;
    margin-bottom: 0.8rem;
}
.stButton>button {
    background: linear-gradient(135deg, #4f46e5, #7c3aed);
    color: white;
    border: none;
    border-radius: 12px;
    padding: .75rem 1rem;
    font-size: 1.05rem;
    font-weight: 600;
    width: 100%;
}
.stButton>button:hover {
    background: linear-gradient(135deg, #3730a3, #6d28d9);
    color: white;
}
</style>
""", unsafe_allow_html=True)


# =========================
# ĐƯỜNG DẪN FILE
# =========================

EXCEL_PATH = "/content/stress_vinedu.xlsx"


# =========================
# TÊN CỘT TRONG FILE
# =========================

ORIGINAL_COLUMNS = [
    "STT",
    "Đối tượng",
    "Thu nhập (Triệu VNĐ/tháng)",
    "Số giờ làm việc / học tập (ngày)",
    "Số giờ ngủ (ngày)",
    "Số ly cà phê (ngày)",
    "Kết quả (Output)",
    "Bối cảnh thực tế tại TP.HCM"
]

RENAME_COLUMNS = {
    "Đối tượng": "doi_tuong",
    "Thu nhập (Triệu VNĐ/tháng)": "thu_nhap",
    "Số giờ làm việc / học tập (ngày)": "gio_lam_hoc",
    "Số giờ ngủ (ngày)": "gio_ngu",
    "Số ly cà phê (ngày)": "ly_ca_phe",
    "Kết quả (Output)": "ket_qua",
    "Bối cảnh thực tế tại TP.HCM": "boi_canh"
}

BASE_FEATURES = [
    "doi_tuong",
    "thu_nhap",
    "gio_lam_hoc",
    "gio_ngu",
    "ly_ca_phe"
]

ENGINEERED_FEATURES = [
    "work_sleep_ratio",
    "sleep_deficit",
    "overwork_flag",
    "low_sleep_flag",
    "high_coffee_flag",
    "coffee_per_work_hour",
    "income_per_work_hour",
    "lifestyle_risk_score"
]

FEATURES = BASE_FEATURES + ENGINEERED_FEATURES

NUMERIC_FEATURES = [
    "thu_nhap",
    "gio_lam_hoc",
    "gio_ngu",
    "ly_ca_phe"
] + ENGINEERED_FEATURES

CATEGORICAL_FEATURES = ["doi_tuong"]

TARGET = "target"


# =========================
# HÀM ĐỌC FILE
# =========================

@st.cache_data
def load_data(uploaded_file=None):
    if uploaded_file is not None:
        file_name = uploaded_file.name.lower()

        if file_name.endswith(".csv"):
            raw_df = pd.read_csv(uploaded_file)
            source = "File CSV vừa upload"
        else:
            raw_df = pd.read_excel(uploaded_file, sheet_name=0, header=2)
            source = "File Excel vừa upload"
    else:
        raw_df = pd.read_excel(EXCEL_PATH, sheet_name=0, header=2)
        source = EXCEL_PATH

    return raw_df, source


def validate_columns(raw_df):
    missing = []

    for col in ORIGINAL_COLUMNS:
        if col not in raw_df.columns:
            missing.append(col)

    return missing


# =========================
# LÀM SẠCH DỮ LIỆU
# =========================

def clean_data(raw_df):
    df = raw_df.copy()

    df = df.rename(columns=RENAME_COLUMNS)

    needed_cols = [
        "doi_tuong",
        "thu_nhap",
        "gio_lam_hoc",
        "gio_ngu",
        "ly_ca_phe",
        "ket_qua",
        "boi_canh"
    ]

    df = df[needed_cols]

    for col in ["thu_nhap", "gio_lam_hoc", "gio_ngu", "ly_ca_phe"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in ["doi_tuong", "ket_qua", "boi_canh"]:
        df[col] = df[col].astype(str).str.strip()

    df = df.dropna(subset=[
        "doi_tuong",
        "thu_nhap",
        "gio_lam_hoc",
        "gio_ngu",
        "ly_ca_phe",
        "ket_qua"
    ])

    df["thu_nhap"] = df["thu_nhap"].clip(lower=0)
    df["gio_lam_hoc"] = df["gio_lam_hoc"].clip(lower=0, upper=24)
    df["gio_ngu"] = df["gio_ngu"].clip(lower=0, upper=24)
    df["ly_ca_phe"] = df["ly_ca_phe"].clip(lower=0)

    df["target"] = df["ket_qua"].apply(
        lambda x: 0 if str(x).strip().lower() == "normal" else 1
    )

    df["nhan_hien_thi"] = df["target"].map({
        0: "Bình thường",
        1: "Căng thẳng / rủi ro cao"
    })

    return df


# =========================
# FEATURE ENGINEERING
# =========================

def add_feature_engineering(df):
    df = df.copy()

    df["work_sleep_ratio"] = df["gio_lam_hoc"] / (df["gio_ngu"] + 0.1)

    df["sleep_deficit"] = np.maximum(0, 7 - df["gio_ngu"])

    df["overwork_flag"] = (df["gio_lam_hoc"] >= 10).astype(int)

    df["low_sleep_flag"] = (df["gio_ngu"] < 6).astype(int)

    df["high_coffee_flag"] = (df["ly_ca_phe"] >= 3).astype(int)

    df["coffee_per_work_hour"] = df["ly_ca_phe"] / (df["gio_lam_hoc"] + 0.1)

    df["income_per_work_hour"] = df["thu_nhap"] / (df["gio_lam_hoc"] + 0.1)

    df["lifestyle_risk_score"] = (
        df["overwork_flag"] * 2
        + df["low_sleep_flag"] * 2
        + df["high_coffee_flag"] * 1
        + np.clip((df["gio_lam_hoc"] - 8) / 4, 0, 2)
        + np.clip((7 - df["gio_ngu"]) / 2, 0, 2)
    )

    return df


# =========================
# TIỀN XỬ LÝ CHO MODEL
# =========================

def make_preprocess():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocess = ColumnTransformer([
        ("num", numeric_pipeline, NUMERIC_FEATURES),
        ("cat", categorical_pipeline, CATEGORICAL_FEATURES)
    ])

    return preprocess


# =========================
# HUẤN LUYỆN MÔ HÌNH
# =========================

@st.cache_resource
def train_models(df_fe):
    X = df_fe[FEATURES]
    y = df_fe[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    models = {
        "Logistic Regression": Pipeline([
            ("preprocess", make_preprocess()),
            ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
        ]),
        "Decision Tree": Pipeline([
            ("preprocess", make_preprocess()),
            ("model", DecisionTreeClassifier(
                max_depth=5,
                random_state=42,
                class_weight="balanced"
            ))
        ]),
        "Random Forest": Pipeline([
            ("preprocess", make_preprocess()),
            ("model", RandomForestClassifier(
                n_estimators=300,
                max_depth=7,
                random_state=42,
                class_weight="balanced"
            ))
        ]),
        "Gradient Boosting": Pipeline([
            ("preprocess", make_preprocess()),
            ("model", GradientBoostingClassifier(
                n_estimators=150,
                learning_rate=0.05,
                max_depth=3,
                random_state=42
            ))
        ]),
        "SVM RBF": Pipeline([
            ("preprocess", make_preprocess()),
            ("model", SVC(
                kernel="rbf",
                C=1.5,
                probability=True,
                random_state=42,
                class_weight="balanced"
            ))
        ]),
        "KNN": Pipeline([
            ("preprocess", make_preprocess()),
            ("model", KNeighborsClassifier(n_neighbors=9))
        ])
    }

    results = {}

    min_class_count = y.value_counts().min()
    n_splits = min(5, int(min_class_count))

    if n_splits < 2:
        n_splits = 2

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    for name, model in models.items():
        model.fit(X_train, y_train)

        pred = model.predict(X_test)
        prob = model.predict_proba(X_test)[:, 1]

        try:
            cv_auc = cross_val_score(
                model,
                X,
                y,
                cv=cv,
                scoring="roc_auc"
            ).mean()
        except:
            cv_auc = np.nan

        results[name] = {
            "model": model,
            "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred, zero_division=0),
            "f1": f1_score(y_test, pred, zero_division=0),
            "auc": roc_auc_score(y_test, prob),
            "cv_auc": cv_auc,
            "y_test": y_test,
            "pred": pred,
            "prob": prob,
            "X_test": X_test
        }

    return results


# =========================
# ĐỌC FILE VÀ CHUẨN BỊ DATA
# =========================

with st.sidebar:
    st.markdown("## ⚙️ Cài đặt")
    uploaded = st.file_uploader(
        "Upload file stress_vinedu.xlsx hoặc CSV mới",
        type=["xlsx", "xls", "csv"]
    )

try:
    raw_df, source = load_data(uploaded)
except Exception as e:
    st.error("Không đọc được file dữ liệu.")
    st.write("Kiểm tra file tại đường dẫn:")
    st.code(EXCEL_PATH)
    st.write("Lỗi chi tiết:")
    st.code(str(e))
    st.stop()

missing_cols = validate_columns(raw_df)

if missing_cols:
    st.error("File thiếu các cột bắt buộc sau:")
    st.write(missing_cols)
    st.markdown("Các cột cần có:")
    st.code(", ".join(ORIGINAL_COLUMNS))
    st.stop()

df_clean = clean_data(raw_df)
df_fe = add_feature_engineering(df_clean)

if len(df_fe) < 30:
    st.error("Dữ liệu quá ít để huấn luyện mô hình.")
    st.stop()

with st.spinner("Đang huấn luyện mô hình Machine Learning..."):
    results = train_models(df_fe)

best_name = max(results, key=lambda x: results[x]["auc"])


# =========================
# SIDEBAR MODEL
# =========================

with st.sidebar:
    st.markdown("## 🤖 Mô hình")

    model_choice = st.selectbox(
        "Chọn mô hình dự đoán:",
        list(results.keys()),
        index=list(results.keys()).index(best_name)
    )

    threshold = st.slider(
        "Ngưỡng phân loại rủi ro cao:",
        min_value=0.10,
        max_value=0.90,
        value=0.50,
        step=0.05
    )

    chosen = results[model_choice]

    st.markdown("---")
    st.markdown("### Chỉ số mô hình")
    st.markdown(f"**Accuracy:** `{chosen['accuracy']:.3f}`")
    st.markdown(f"**Precision:** `{chosen['precision']:.3f}`")
    st.markdown(f"**Recall:** `{chosen['recall']:.3f}`")
    st.markdown(f"**F1-score:** `{chosen['f1']:.3f}`")
    st.markdown(f"**ROC AUC:** `{chosen['auc']:.3f}`")

    if pd.notna(chosen["cv_auc"]):
        st.markdown(f"**CV AUC:** `{chosen['cv_auc']:.3f}`")
    else:
        st.markdown("**CV AUC:** `Không tính được`")

    st.markdown("---")
    st.markdown("### Dữ liệu")
    st.markdown(f"**Nguồn:** `{source}`")
    st.markdown(f"**Số dòng:** `{len(df_fe)}`")
    st.markdown(f"**Số biến đầu vào:** `{len(FEATURES)}`")


# =========================
# HEADER
# =========================

st.markdown("""
<div class="main-header">
    <h1>🧠 Ứng Dụng Dự Đoán Mức Độ Căng Thẳng</h1>
    <p>Phân tích dữ liệu khảo sát, Feature Engineering, EDA và Machine Learning</p>
</div>
""", unsafe_allow_html=True)

st.info(
    "Ứng dụng sử dụng dữ liệu khảo sát, tạo thêm đặc trưng mới và huấn luyện mô hình để phân loại trạng thái bình thường hoặc rủi ro cao."
)


# =========================
# TABS
# =========================

tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "🔮 Dự đoán",
    "📊 EDA",
    "🧩 Feature Engineering",
    "🏆 So sánh mô hình",
    "📁 Kiểm tra dữ liệu"
])


# =========================
# TAB 1: DỰ ĐOÁN
# =========================

with tab1:
    left, right = st.columns([1.1, 1], gap="large")

    with left:
        st.markdown("## Nhập thông tin")

        doi_tuong_options = sorted(df_fe["doi_tuong"].dropna().unique().tolist())

        c1, c2 = st.columns(2)

        with c1:
            doi_tuong = st.selectbox(
                "Đối tượng",
                doi_tuong_options
            )

            thu_nhap = st.slider(
                "Thu nhập (triệu VNĐ/tháng)",
                min_value=0.0,
                max_value=float(max(40, df_fe["thu_nhap"].max())),
                value=float(df_fe["thu_nhap"].median()),
                step=0.5
            )

            gio_lam_hoc = st.slider(
                "Số giờ làm việc / học tập mỗi ngày",
                min_value=0.0,
                max_value=16.0,
                value=8.0,
                step=0.1
            )

        with c2:
            gio_ngu = st.slider(
                "Số giờ ngủ mỗi ngày",
                min_value=0.0,
                max_value=12.0,
                value=7.0,
                step=0.1
            )

            ly_ca_phe = st.slider(
                "Số ly cà phê mỗi ngày",
                min_value=0,
                max_value=10,
                value=2,
                step=1
            )

        predict_btn = st.button("🔍 Dự đoán mức độ căng thẳng")

    with right:
        st.markdown("## Kết quả dự đoán")

        if predict_btn:
            input_raw = pd.DataFrame([{
                "doi_tuong": doi_tuong,
                "thu_nhap": thu_nhap,
                "gio_lam_hoc": gio_lam_hoc,
                "gio_ngu": gio_ngu,
                "ly_ca_phe": ly_ca_phe,
                "ket_qua": "Unknown",
                "boi_canh": ""
            }])

            input_fe = add_feature_engineering(input_raw)
            input_X = input_fe[FEATURES]

            model = results[model_choice]["model"]
            prob_high = model.predict_proba(input_X)[0][1]
            percent_high = prob_high * 100
            risk_class = 1 if prob_high >= threshold else 0

            if risk_class == 1:
                box_class = "result-high"
                title = "Căng thẳng / rủi ro cao"
                color = "#b91c1c"
                icon = "⚠️"
                note = "Mô hình xếp trường hợp này vào nhóm rủi ro cao dựa trên các chỉ số đầu vào."
            else:
                box_class = "result-low"
                title = "Bình thường"
                color = "#15803d"
                icon = "✅"
                note = "Mô hình xếp trường hợp này vào nhóm bình thường dựa trên các chỉ số đầu vào."

            st.markdown(f"""
            <div class="{box_class}">
                <p style="font-size:1.2rem;margin-bottom:0.2rem;">{icon} Kết luận</p>
                <h2 style="color:{color};font-size:2.4rem;margin:0;">{title}</h2>
                <h1 style="color:{color};font-size:4rem;margin:0.5rem 0;">{percent_high:.1f}%</h1>
                <p style="font-size:1rem;color:#334155;">Xác suất thuộc nhóm căng thẳng/rủi ro cao</p>
                <p style="font-size:.9rem;color:#475569;">Mô hình: {model_choice}</p>
            </div>
            """, unsafe_allow_html=True)

            st.info(note)

            st.markdown("### Phân tích nhanh")

            tips = []

            if gio_lam_hoc >= 10:
                tips.append("Số giờ làm việc/học tập cao là yếu tố làm tăng điểm rủi ro.")

            if gio_ngu < 6:
                tips.append("Số giờ ngủ thấp làm tăng biến sleep_deficit và lifestyle_risk_score.")

            if ly_ca_phe >= 3:
                tips.append("Số ly cà phê cao làm tăng biến high_coffee_flag.")

            if input_fe["work_sleep_ratio"].iloc[0] >= 1.6:
                tips.append("Tỉ lệ làm việc/học tập so với ngủ khá cao.")

            if not tips:
                tips.append("Các chỉ số đầu vào tương đối cân bằng theo các ngưỡng của app.")

            for t in tips:
                st.markdown(f"- {t}")

            normal_df = df_fe[df_fe[TARGET] == 0]

            if len(normal_df) > 0:
                representative = {
                    "doi_tuong": normal_df["doi_tuong"].mode()[0],
                    "thu_nhap": normal_df["thu_nhap"].median(),
                    "gio_lam_hoc": normal_df["gio_lam_hoc"].median(),
                    "gio_ngu": normal_df["gio_ngu"].median(),
                    "ly_ca_phe": normal_df["ly_ca_phe"].median(),
                    "ket_qua": "Normal",
                    "boi_canh": ""
                }
            else:
                representative = {
                    "doi_tuong": df_fe["doi_tuong"].mode()[0],
                    "thu_nhap": df_fe["thu_nhap"].median(),
                    "gio_lam_hoc": df_fe["gio_lam_hoc"].median(),
                    "gio_ngu": df_fe["gio_ngu"].median(),
                    "ly_ca_phe": df_fe["ly_ca_phe"].median(),
                    "ket_qua": "Normal",
                    "boi_canh": ""
                }

            normal_input = pd.DataFrame([representative])
            normal_fe = add_feature_engineering(normal_input)
            normal_prob = model.predict_proba(normal_fe[FEATURES])[0][1]
            normal_percent = normal_prob * 100

            st.markdown("### So sánh với hồ sơ bình thường đại diện")

            ca, cb = st.columns(2)

            with ca:
                st.metric("Người đang kiểm tra", f"{percent_high:.1f}%")

            with cb:
                st.metric("Hồ sơ bình thường", f"{normal_percent:.1f}%")

            diff = percent_high - normal_percent

            if diff > 0:
                st.warning(f"Người đang kiểm tra cao hơn hồ sơ bình thường khoảng {diff:.1f} điểm phần trăm.")
            elif diff < 0:
                st.success(f"Người đang kiểm tra thấp hơn hồ sơ bình thường khoảng {abs(diff):.1f} điểm phần trăm.")
            else:
                st.info("Hai hồ sơ có xác suất gần như bằng nhau.")

            radar_labels = ["Làm/Học", "Thiếu ngủ", "Cà phê", "Tỉ lệ làm-ngủ", "Risk score"]

            current_values = [
                min(gio_lam_hoc / 16, 1),
                min(input_fe["sleep_deficit"].iloc[0] / 7, 1),
                min(ly_ca_phe / 10, 1),
                min(input_fe["work_sleep_ratio"].iloc[0] / 4, 1),
                min(input_fe["lifestyle_risk_score"].iloc[0] / 8, 1)
            ]

            normal_values = [
                min(normal_fe["gio_lam_hoc"].iloc[0] / 16, 1),
                min(normal_fe["sleep_deficit"].iloc[0] / 7, 1),
                min(normal_fe["ly_ca_phe"].iloc[0] / 10, 1),
                min(normal_fe["work_sleep_ratio"].iloc[0] / 4, 1),
                min(normal_fe["lifestyle_risk_score"].iloc[0] / 8, 1)
            ]

            angles = np.linspace(0, 2 * np.pi, len(radar_labels), endpoint=False).tolist()

            current_values += current_values[:1]
            normal_values += normal_values[:1]
            angles += angles[:1]

            fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

            ax.plot(angles, current_values, "o-", linewidth=2, color="#2563eb", label="Người đang kiểm tra")
            ax.fill(angles, current_values, alpha=0.22, color="#2563eb")

            ax.plot(angles, normal_values, "o-", linewidth=2, color="#9333ea", label="Hồ sơ bình thường")
            ax.fill(angles, normal_values, alpha=0.25, color="#9333ea")

            ax.set_thetagrids(np.degrees(angles[:-1]), radar_labels)
            ax.set_ylim(0, 1)
            ax.set_title("So sánh hồ sơ căng thẳng", fontweight="bold", pad=20)
            ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15))

            fig.tight_layout()
            st.pyplot(fig, use_container_width=True)
            plt.close()

        else:
            st.info("Nhập thông tin và bấm nút dự đoán.")


# =========================
# TAB 2: EDA
# =========================

with tab2:
    st.markdown("## EDA - Phân tích dữ liệu khám phá")

    m1, m2, m3, m4 = st.columns(4)

    high_rate = df_fe[TARGET].mean() * 100

    m1.metric("Số dòng dữ liệu", len(df_fe))
    m2.metric("Số đối tượng", df_fe["doi_tuong"].nunique())
    m3.metric("Tỉ lệ rủi ro cao", f"{high_rate:.1f}%")
    m4.metric("Số feature sau FE", len(FEATURES))

    st.markdown("---")

    c1, c2 = st.columns(2)

    with c1:
        st.markdown("### Phân bố kết quả")
        count_data = df_fe["nhan_hien_thi"].value_counts()

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(count_data.index, count_data.values)
        ax.set_ylabel("Số lượng")
        ax.set_title("Phân bố nhãn đầu ra", fontweight="bold")
        for i, v in enumerate(count_data.values):
            ax.text(i, v + 2, str(v), ha="center", fontweight="bold")
        fig.tight_layout()
        st.pyplot(fig, use_container_width=True)
        plt.close()

    with c2:
        st.markdown("### Phân bố đối tượng khảo sát")
        obj_count = df_fe["doi_tuong"].value_counts()

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(obj_count.index, obj_count.values)
        ax.set_ylabel("Số lượng")
        ax.set_title("Số lượng theo đối tượng", fontweight="bold")
        ax.tick_params(axis="x", rotation=20)
        fig.tight_layout()
        st.pyplot(fig, use_container_width=True)
        plt.close()

    c3, c4 = st.columns(2)

    with c3:
        st.markdown("### Giờ làm/học theo kết quả")
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.boxplot(data=df_fe, x="nhan_hien_thi", y="gio_lam_hoc", ax=ax)
        ax.set_xlabel("")
        ax.set_ylabel("Giờ làm/học")
        ax.set_title("Giờ làm/học và nhãn đầu ra", fontweight="bold")
        ax.tick_params(axis="x", rotation=10)
        fig.tight_layout()
        st.pyplot(fig, use_container_width=True)
        plt.close()

    with c4:
        st.markdown("### Giờ ngủ theo kết quả")
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.boxplot(data=df_fe, x="nhan_hien_thi", y="gio_ngu", ax=ax)
        ax.set_xlabel("")
        ax.set_ylabel("Giờ ngủ")
        ax.set_title("Giờ ngủ và nhãn đầu ra", fontweight="bold")
        ax.tick_params(axis="x", rotation=10)
        fig.tight_layout()
        st.pyplot(fig, use_container_width=True)
        plt.close()

    c5, c6 = st.columns(2)

    with c5:
        st.markdown("### Làm/học và ngủ")
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.scatterplot(
            data=df_fe,
            x="gio_lam_hoc",
            y="gio_ngu",
            hue="nhan_hien_thi",
            ax=ax
        )
        ax.set_title("Quan hệ giữa giờ làm/học và giờ ngủ", fontweight="bold")
        fig.tight_layout()
        st.pyplot(fig, use_container_width=True)
        plt.close()

    with c6:
        st.markdown("### Tỉ lệ rủi ro cao theo đối tượng")
        rate_by_obj = df_fe.groupby("doi_tuong")[TARGET].mean().sort_values(ascending=False) * 100

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(rate_by_obj.index, rate_by_obj.values)
        ax.set_ylabel("Tỉ lệ rủi ro cao (%)")
        ax.set_title("Tỉ lệ rủi ro cao theo đối tượng", fontweight="bold")
        ax.tick_params(axis="x", rotation=20)
        for i, v in enumerate(rate_by_obj.values):
            ax.text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=9)
        fig.tight_layout()
        st.pyplot(fig, use_container_width=True)
        plt.close()

    st.markdown("### Ma trận tương quan")

    corr_cols = [
        "thu_nhap",
        "gio_lam_hoc",
        "gio_ngu",
        "ly_ca_phe"
    ] + ENGINEERED_FEATURES + [TARGET]

    corr = df_fe[corr_cols].corr()

    fig, ax = plt.subplots(figsize=(11, 7))
    sns.heatmap(
        corr,
        annot=True,
        fmt=".2f",
        cmap="RdYlGn",
        center=0,
        ax=ax
    )
    ax.set_title("Correlation Matrix sau Feature Engineering", fontweight="bold")
    fig.tight_layout()
    st.pyplot(fig, use_container_width=True)
    plt.close()


# =========================
# TAB 3: FEATURE ENGINEERING
# =========================

with tab3:
    st.markdown("## Feature Engineering")

    st.markdown("""
    <div class="feature-card">
        Feature Engineering là bước tạo thêm biến mới từ dữ liệu gốc.
        Trong app này, các biến mới được tạo từ giờ làm/học, giờ ngủ, cà phê và thu nhập.
    </div>
    """, unsafe_allow_html=True)

    fe_info = pd.DataFrame({
        "Feature mới": [
            "work_sleep_ratio",
            "sleep_deficit",
            "overwork_flag",
            "low_sleep_flag",
            "high_coffee_flag",
            "coffee_per_work_hour",
            "income_per_work_hour",
            "lifestyle_risk_score"
        ],
        "Ý nghĩa": [
            "Tỉ lệ giữa số giờ làm/học và số giờ ngủ",
            "Số giờ thiếu ngủ so với mốc 7 giờ",
            "Bằng 1 nếu làm/học từ 10 giờ trở lên",
            "Bằng 1 nếu ngủ dưới 6 giờ",
            "Bằng 1 nếu uống từ 3 ly cà phê trở lên",
            "Số ly cà phê chia cho số giờ làm/học",
            "Thu nhập chia cho số giờ làm/học",
            "Điểm tổng hợp từ quá tải, thiếu ngủ và cà phê"
        ]
    })

    st.dataframe(fe_info, use_container_width=True, hide_index=True)

    st.markdown("### Dữ liệu sau khi thêm feature mới")
    show_cols = BASE_FEATURES + ENGINEERED_FEATURES + ["nhan_hien_thi"]
    st.dataframe(df_fe[show_cols].head(30), use_container_width=True)

    c1, c2 = st.columns(2)

    with c1:
        st.markdown("### Lifestyle Risk Score")
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.hist(df_fe["lifestyle_risk_score"], bins=20, edgecolor="white", alpha=0.85)
        ax.set_title("Phân bố lifestyle_risk_score", fontweight="bold")
        ax.set_xlabel("Risk score")
        ax.set_ylabel("Tần suất")
        fig.tight_layout()
        st.pyplot(fig, use_container_width=True)
        plt.close()

    with c2:
        st.markdown("### Risk score theo kết quả")
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.boxplot(data=df_fe, x="nhan_hien_thi", y="lifestyle_risk_score", ax=ax)
        ax.set_xlabel("")
        ax.set_ylabel("Lifestyle risk score")
        ax.set_title("Feature mới và nhãn đầu ra", fontweight="bold")
        ax.tick_params(axis="x", rotation=10)
        fig.tight_layout()
        st.pyplot(fig, use_container_width=True)
        plt.close()


# =========================
# TAB 4: SO SÁNH MÔ HÌNH
# =========================

with tab4:
    st.markdown("## So sánh mô hình Machine Learning")

    rows = []

    for name, r in results.items():
        rows.append({
            "Mô hình": name,
            "Accuracy": round(r["accuracy"], 4),
            "Precision": round(r["precision"], 4),
            "Recall": round(r["recall"], 4),
            "F1-score": round(r["f1"], 4),
            "ROC AUC": round(r["auc"], 4),
            "CV AUC": round(r["cv_auc"], 4) if pd.notna(r["cv_auc"]) else None
        })

    comp_df = pd.DataFrame(rows).sort_values("ROC AUC", ascending=False)

    def highlight_best(s):
        is_best = s == s.max()
        return ["background-color:#ddd6fe;font-weight:bold" if v else "" for v in is_best]

    st.dataframe(
        comp_df.style.apply(
            highlight_best,
            subset=["Accuracy", "Precision", "Recall", "F1-score", "ROC AUC"]
        ),
        use_container_width=True,
        hide_index=True
    )

    c1, c2 = st.columns(2)

    with c1:
        st.markdown("### Biểu đồ ROC AUC")
        names = comp_df["Mô hình"].tolist()
        aucs = comp_df["ROC AUC"].tolist()

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.barh(names, aucs)
        ax.set_xlabel("ROC AUC")
        ax.set_title("So sánh ROC AUC", fontweight="bold")
        ax.set_xlim(0, 1)
        for i, v in enumerate(aucs):
            ax.text(v + 0.01, i, f"{v:.3f}", va="center")
        fig.tight_layout()
        st.pyplot(fig, use_container_width=True)
        plt.close()

    with c2:
        st.markdown("### Ma trận nhầm lẫn")
        chosen = results[model_choice]
        cm = confusion_matrix(chosen["y_test"], chosen["pred"])

        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Purples",
            xticklabels=["Dự đoán bình thường", "Dự đoán rủi ro"],
            yticklabels=["Thực tế bình thường", "Thực tế rủi ro"],
            ax=ax
        )
        ax.set_title(f"Confusion Matrix - {model_choice}", fontweight="bold")
        fig.tight_layout()
        st.pyplot(fig, use_container_width=True)
        plt.close()

    st.markdown("### ROC Curve")

    fig, ax = plt.subplots(figsize=(7, 5))

    for name, r in results.items():
        fpr, tpr, _ = roc_curve(r["y_test"], r["prob"])
        ax.plot(fpr, tpr, label=f"{name} AUC={r['auc']:.3f}")

    ax.plot([0, 1], [0, 1], linestyle="--", label="Random")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve", fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    st.pyplot(fig, use_container_width=True)
    plt.close()

    st.markdown("### Mức độ quan trọng của feature")

    try:
        selected_model = results[model_choice]["model"]
        X_test = results[model_choice]["X_test"]
        y_test = results[model_choice]["y_test"]

        perm = permutation_importance(
            selected_model,
            X_test,
            y_test,
            n_repeats=5,
            random_state=42,
            scoring="f1"
        )

        imp_df = pd.DataFrame({
            "Feature": FEATURES,
            "Importance": perm.importances_mean
        }).sort_values("Importance", ascending=True)

        fig, ax = plt.subplots(figsize=(7, 5))
        ax.barh(imp_df["Feature"], imp_df["Importance"])
        ax.set_title(f"Permutation Importance - {model_choice}", fontweight="bold")
        ax.set_xlabel("Importance")
        fig.tight_layout()
        st.pyplot(fig, use_container_width=True)
        plt.close()

    except Exception as e:
        st.info("Không tính được feature importance cho mô hình hiện tại.")
        st.code(str(e))

    st.info(f"Mô hình tốt nhất theo ROC AUC hiện tại là: {best_name}")


# =========================
# TAB 5: KIỂM TRA DỮ LIỆU
# =========================

with tab5:
    st.markdown("## Kiểm tra dữ liệu")

    st.markdown("### Nguồn dữ liệu")
    st.code(source)

    st.markdown("### Dữ liệu gốc")
    st.dataframe(raw_df.head(20), use_container_width=True)

    st.markdown("### Dữ liệu sau khi làm sạch")
    st.dataframe(df_clean.head(30), use_container_width=True)

    st.markdown("### Dữ liệu sau Feature Engineering")
    st.dataframe(df_fe.head(30), use_container_width=True)

    st.markdown("### Thống kê mô tả")
    describe_cols = [
        "thu_nhap",
        "gio_lam_hoc",
        "gio_ngu",
        "ly_ca_phe"
    ] + ENGINEERED_FEATURES

    st.dataframe(
        df_fe[describe_cols].describe().T,
        use_container_width=True
    )

    st.download_button(
        label="Tải dữ liệu đã xử lý",
        data=df_fe.to_csv(index=False).encode("utf-8-sig"),
        file_name="stress_vinedu_processed.csv",
        mime="text/csv"
    )

Writing streamlit_app.py


In [4]:
from pyngrok import ngrok
import subprocess
import time
import getpass
import os

file_path = "/content/stress_vinedu.xlsx"

if not os.path.exists(file_path):
    print("Chưa tìm thấy file:", file_path)
    print("Hãy upload stress_vinedu.xlsx vào Colab trước khi chạy app.")
else:
    print("Đã tìm thấy file:", file_path)

NGROK_TOKEN = getpass.getpass("Nhập ngrok token: ")

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

ngrok.kill()

proc = subprocess.Popen(
    [
        "streamlit", "run", "streamlit_app.py",
        "--server.port=8501",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

public_url = ngrok.connect(8501)
print("Link app của bạn:")
print(public_url)

Đã tìm thấy file: /content/stress_vinedu.xlsx
Nhập ngrok token: ··········
Link app của bạn:
NgrokTunnel: "https://bruising-slideshow-chapped.ngrok-free.dev" -> "http://localhost:8501"
